# 17 — Embedding design amplifies the effect

In `04/16` the equal-weight qutrit read the second harmonic, and needed longer records than the second-harmonic PLV to do it. This notebook turns a dial the capstone left at its default, the **weights** of the harmonics, and measures what changes. The gap between the two ensembles of `04/15` grows about threefold. With the first harmonic turned down to a weight of $0.2$, the detection rate reaches $0.88$ with 100 samples per ensemble, where the equal-weight measure is still at $0.58$ with 20,000. Then you will see why it works, and why the gain stops where the matching classical statistic already stands.

**Prerequisites:** `15_discriminating_experiment` (the matched-PLV ensembles, the area under the curve and the detection rate), `16_capstone_synthesis` (the conclusion of the capstone and the embedding-as-dial idea), `14_richer_embedding_applied` (the multi-frequency qutrit).

**What you'll be able to do**
- Use `weighted_multifreq_state` to set *how strongly* each circular moment is encoded into $\rho_{AB}$.
- Score a weighting with the protocol of `04/15`, beside the classical statistic of the same moment.
- Explain, from the long-run response of the measure to each moment, why turning the first harmonic down helps, and what it turns the measure into.
- Score another lever the same way: the *amplitude* embedding, which reaches phase–amplitude coupling.
- State what embedding design bought here: a measure that needs fewer samples, and in no case one that ranks the records better than the classical statistic of the same moment.

In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.infotheory.classical import kl_divergence
from qot_course.quantum_ot.bench_protocol import (
    auc,
    detection_rate,
    harmonic_plv,
    long_run_statistics,
)
from qot_course.quantum_ot.capstone import plv, coupling_qmi
from qot_course.quantum_ot.embeddings import (
    weighted_multifreq_state,
    joint_density_from_states,
    amplitude_phase_state,
)
from qot_course.quantum_ot.discrimination import matched_plv_ensembles, sample_phase_difference

np.random.seed(42)
viz.use_course_style()

# Sample count per channel: large enough that the circular moments are tight Monte-Carlo
# estimates (error ~ 1/sqrt(N)), matching the discriminating experiment of notebook 15.
N = 150_000

## 1. The dial we left at its default

The multi-frequency qutrit of `04/14` is an *equal-weight* superposition: $\tfrac{1}{\sqrt 3}\bigl(|0\rangle + e^{i\theta}|1\rangle + e^{2i\theta}|2\rangle\bigr)$. Each level carries the same amplitude, so the joint density $\rho_{AB}$ gives the first circular moment of the phase difference (the one PLV reads) and the second (the one that differs between our two ensembles) comparable room.

In the matched-PLV experiment the first moment is *matched* by construction. It carries no difference between the two ensembles, yet it still wobbles from record to record. What if we turned it down? `weighted_multifreq_state` sets the weights $(w_0, w_1, w_2)$ of the three levels, and rescales them so that the state keeps unit length.

To see what $w_1$ controls, look at how a moment enters $\rho_{AB}$. An entry of $\rho_{AB}$ averages a product of four amplitudes, two from each system. It carries the first moment when each system contributes two neighbouring levels, $(0, 1)$ or $(1, 2)$. Level 1 is in every such pair, so each of these entries has a factor $w_1^2$. The second moment enters when each system contributes the pair $(0, 2)$, with a factor $w_0^2 w_2^2$ and no $w_1$ at all. Lowering $w_1$ therefore quietens the first moment and leaves the room to the second.

We rebuild the two ensembles of `04/15` (first moment $0.4$ in both, second moment $0.0$ against $0.3$) and try three weightings: the equal-weight default, then $(1, 0.5, 2)$ and $(1, 0.2, 2)$, the two carried by the bench files of `04/16`. The weight 2 on the top level is part of that choice, not a requirement.

In [ ]:
# The two ensembles from notebook 15: identical first moment a1 (-> identical PLV),
# different second moment a2 (0.0 against 0.3).
(ta_lo, tb_lo), (ta_hi, tb_hi) = matched_plv_ensembles(
    N, a1=0.4, a2_low=0.0, a2_high=0.3, seed=0
)
print(f"PLV gap (matched by construction): {abs(plv(ta_lo, tb_lo) - plv(ta_hi, tb_hi)):.4f}")


def weighted_qmi(ta, tb, weights, harmonics=(1, 2)):
    """Quantum mutual information (nats) of the weighted multi-frequency joint embedding."""
    dim = 1 + len(harmonics)
    rho = joint_density_from_states(
        weighted_multifreq_state(ta, harmonics, weights),
        weighted_multifreq_state(tb, harmonics, weights),
    )
    return coupling_qmi(rho, dims=(dim, dim))


configs = [
    ("equal weights (1, 1, 1)", (1.0, 1.0, 1.0)),
    ("first harmonic at 0.5: (1, 0.5, 2)", (1.0, 0.5, 2.0)),
    ("first harmonic at 0.2: (1, 0.2, 2)", (1.0, 0.2, 2.0)),
]
gaps = []
print(f"{'weights':<38}{'low':>10}{'high':>10}{'gap':>10}   (nats)")
for name, w in configs:
    low, high = weighted_qmi(ta_lo, tb_lo, w), weighted_qmi(ta_hi, tb_hi, w)
    gaps.append(high - low)
    print(f"{name:<38}{low:>10.5f}{high:>10.5f}{high - low:>10.5f}")

**Read the output.** The first line confirms the construction: the two ensembles have the same PLV to within $0.002$. Then, for each weighting, the quantum mutual information (QMI) of the low ensemble, of the high ensemble, and their gap. Two things move as the first-harmonic weight goes down. The values themselves shrink, from $0.074$ to $0.001$ nat for the low ensemble, because most of the equal-weight QMI came from the first moment. And the gap grows, from $0.0044$ to $0.0145$ nat. The figure sets these gaps beside what a quantum processor could resolve.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.8))
y = np.arange(len(configs))[::-1]
bar_colors = [COLORS["muted"], COLORS["quantum"], COLORS["highlight"]]
ax.barh(y, gaps, color=bar_colors, edgecolor="white", linewidth=1.2, zorder=3)
# What a real device would have to resolve. A gap is a DIFFERENCE of two QMI estimates; if the
# two are independent and equally noisy, its statistical band is sqrt(2) * sigma_stat.
# Notebook 05/06 reports two values of sigma_stat for a single estimate: 0.012 nats on the
# real processor (the line drawn here) and 0.0128 nats on the offline device model.
HARDWARE_SIGMA_STAT = 0.012   # real run reported in 05/06
MODEL_SIGMA_STAT = 0.0128     # offline device model of 05/06
hardware_band = np.sqrt(2) * HARDWARE_SIGMA_STAT
model_band = np.sqrt(2) * MODEL_SIGMA_STAT
ax.axvline(hardware_band, color=COLORS["negative"], lw=1.6, ls="--", zorder=2,
           label=f"one band of the real device ({hardware_band:.3f} nat)")
for yi, g in zip(y, gaps):
    ax.annotate(f"{g:.4f}", (g, yi), xytext=(6, 0), textcoords="offset points",
                va="center", ha="left", fontsize=10, color=COLORS["text"])
ax.set_yticks(y)
ax.set_yticklabels([name for name, _ in configs])
ax.set_xlabel("QMI gap between the two matched-PLV ensembles  (nats)")
ax.set_xlim(0, max(max(gaps) * 1.3, hardware_band * 1.1))
ax.set_title("Turning the first harmonic down enlarges the gap", pad=10)
ax.legend(loc="upper right", fontsize=9)
ax.grid(axis="y", visible=False)
fig.tight_layout()
plt.show()
for (name, _), g in zip(configs, gaps):
    print(f"{name:<38} gap / band:  real device = {g / hardware_band:.2f}   "
          f"offline model = {g / model_band:.2f}")

**Read the figure.** Three weightings of the *same* qutrit embedding, applied to the *same* two ensembles. The grey bar is the equal-weight default of `04/15`: a gap of $0.0044$ nat, about a quarter of the way to the dashed line.

That line is one **band** of a real device: the statistical scatter, one standard deviation caused by the finite number of shots, of a *difference* of two mutual informations. Notebook `05/06` measures $0.012$ nat for a single estimate on the real processor. A difference of two estimates that are independent and equally noisy scatters $\sqrt 2$ times more, so $0.017$ nat.

As the first-harmonic weight goes down (blue, then rose), the gap climbs: the rose bar is about three times longer than the grey one, at $0.85$ of that band, or $0.80$ bands of the offline device model of `05/06`, whose scatter is about the same. Both ratios are printed under the figure. Either way the weighted gap is a little under one band: a real gain, and not yet a detection, for which the usual rule of thumb asks two or three bands. Nothing about the *systems* changed. We changed how the measure spends its sensitivity.

## 2. Does a larger gap mean fewer samples?

A gap is a statement about very long records. What you hold in practice is one record of finite length, and `04/15` gave you two ways to say whether a statistic tells two ensembles apart at that length. The **area under the curve** (AUC) is the probability that a high ensemble scores above a low one: $0.5$ is chance, $1$ is perfect ranking. The **detection rate** is the share of high-minus-low differences that fall outside the band holding 95 % of the low-minus-low differences.

We run that protocol, a hundred ensembles per set and five repetitions, on six statistics:

- the **second-harmonic PLV** $|\langle e^{2i\Delta\theta}\rangle|$, the classical statistic of the moment that differs;
- its **square**, a check whose purpose the second reading below explains;
- the QMI of the three weightings of section 1, and of a fourth, $(1, 0, 2)$, in which the first harmonic is switched off entirely.

In [ ]:
STATISTICS = {
    "second-harmonic PLV": lambda ta, tb: harmonic_plv(ta, tb, harmonic=2),
    "same, squared": lambda ta, tb: harmonic_plv(ta, tb, harmonic=2) ** 2,
    "QMI (1, 1, 1)": lambda ta, tb: weighted_qmi(ta, tb, (1.0, 1.0, 1.0)),
    "QMI (1, 0.5, 2)": lambda ta, tb: weighted_qmi(ta, tb, (1.0, 0.5, 2.0)),
    "QMI (1, 0.2, 2)": lambda ta, tb: weighted_qmi(ta, tb, (1.0, 0.2, 2.0)),
    "QMI (1, 0, 2)": lambda ta, tb: weighted_qmi(ta, tb, (1.0, 0.0, 2.0)),
}
N_RECORDS = 100  # ensembles per set, as in notebook 15
N_SEEDS = 5      # independent repetitions of the whole comparison
SIZES = [50, 100, 200, 2_000, 20_000]  # samples per ensemble
FALSE_ALARM_RATE = 0.05  # fraction of null differences allowed outside the detection band

# matched_plv_ensembles consumes seed, seed + 1 and seed + 2. Every call gets its own multiple
# of 10, drawn without replacement, so no two calls share a random stream.
seed_table = 10 * np.random.default_rng(17).choice(
    10**8, size=(len(SIZES), N_SEEDS, 2, N_RECORDS), replace=False
)


def run_comparison(n, seeds_observed, seeds_null, statistics):
    """Protocol of notebook 15: N_RECORDS (low, high) pairs and N_RECORDS (low, low) pairs.

    Returns, for each statistic, the area under the curve (probability that a high ensemble
    scores above a low one) and the detection rate (share of high - low differences outside
    the band that holds 95 % of the low - low differences).
    """
    values = {name: np.empty((4, N_RECORDS)) for name in statistics}  # low, high, null 1, null 2
    for i in range(N_RECORDS):
        low, high = matched_plv_ensembles(
            n, a1=0.4, a2_low=0.0, a2_high=0.3, seed=int(seeds_observed[i])
        )
        null_1, null_2 = matched_plv_ensembles(
            n, a1=0.4, a2_low=0.0, a2_high=0.0, seed=int(seeds_null[i])
        )
        for name, statistic in statistics.items():
            values[name][:, i] = [statistic(*pair) for pair in (low, high, null_1, null_2)]
    return {
        name: (
            auc(v[0], v[1]),
            detection_rate(v[1] - v[0], v[3] - v[2], false_alarm_rate=FALSE_ALARM_RATE),
        )
        for name, v in values.items()
    }


def print_tables(summary, sizes, title):
    """Print mean [min, max] over repetitions, one row per statistic and one column per size."""
    names = list(next(iter(summary.values())))
    for column, label in ((1, "detection rate"), (0, "area under the curve")):
        print(f"\n{title}: {label}, mean [min, max] over {N_SEEDS} repetitions")
        print(f"{'samples per ensemble':<22}" + "".join(f"{n:>19,}" for n in sizes))
        for name in names:
            cells = [summary[n][name][:, column] for n in sizes]
            print(f"{name:<22}" + "".join(
                f"{f'{c.mean():.2f} [{c.min():.2f}, {c.max():.2f}]':>19}" for c in cells
            ))


summary = {}
for k, n in enumerate(SIZES):
    start = time.perf_counter()
    runs = [
        run_comparison(n, seed_table[k, s, 0], seed_table[k, s, 1], STATISTICS)
        for s in range(N_SEEDS)
    ]
    summary[n] = {name: np.array([run[name] for run in runs]) for name in STATISTICS}
    print(f"{n:>7,} samples per ensemble: {N_SEEDS} repetitions done "
          f"in {time.perf_counter() - start:5.1f} s")
print_tables(summary, SIZES, "weightings")

**Read the output.** Each cell is a mean over five repetitions, with the lowest and highest repetition in brackets. Start with the first table, the detection rate, and read it row by row.

The equal-weight QMI (third row) is at $0.15$ in the 2,000-sample column and still at $0.58$ in the 20,000-sample column. With the first-harmonic weight at $0.5$ (fourth row) the rate is $0.80$ at 200 samples. At $0.2$ (fifth row) it is $0.88$ at 100 samples and $1.00$ at 200. The weighted measure does better with 100 samples than the equal-weight one does with 20,000. On this pair of ensembles, turning one weight down divided the number of samples needed by at least two hundred. That is the effect the title promises, and it is large.

Now compare with the classical row at the top. The second-harmonic PLV is at $0.90$ with 100 samples and $0.99$ with 200: the weighted QMI has caught up with it. Has it gone past? The two rows are too close to answer at a glance, and the next cell shows you how to read such a near-tie.

**How to read a near-tie.** Go to the 50-sample column of both tables, where no statistic is saturated. By AUC (second table) the classical statistic is level or slightly ahead: $0.96$, with repetitions between $0.95$ and $0.97$, against $0.94$, between $0.92$ and $0.95$, for the weighting $(1, 0.2, 2)$. By detection rate (first table) the two are level: $0.57$ against $0.58$.

The squared row tells you which criterion to trust. Squaring cannot change the order of two values, so the AUC of the square is identical to that of the PLV itself. Its detection rate, though, is *higher*: $0.67$ against $0.57$. Nothing was added to the statistic. The detection rate is computed on differences, and a change of scale that compresses small values shrinks the differences between two low ensembles more than a high-minus-low difference. A statistic can gain detection rate from its scale alone; its AUC cannot move that way. In a near-tie, read the AUC first.

The weighted QMI has such a scale: near zero it grows like the *square* of the second moment, which section 3 checks. In this table the gain does not show. In the bench files of `04/16` it does: the $(1, 0.2, 2)$ embedding has a detection rate a little above that of the second-harmonic PLV in most cells, with an AUC level or a little below. That is the pattern the second exercise of `04/16` let you find. Read with the AUC first, it says: caught up, not ahead.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), sharex=True)

styles = {
    "second-harmonic PLV": (COLORS["source"], "-"),
    "QMI (1, 1, 1)": (COLORS["muted"], "-"),
    "QMI (1, 0.5, 2)": (COLORS["quantum"], "-"),
    "QMI (1, 0.2, 2)": (COLORS["highlight"], "-"),
    "QMI (1, 0, 2)": (COLORS["text"], "--"),
}
POWER_CONVENTION = 0.8  # usual convention for statistical power, not derived from this problem
panels = [(0, "area under the curve", 0.5, "chance"),
          (1, "detection rate", POWER_CONVENTION, "0.8, the usual power convention")]
for ax, (column, ylabel, reference, reference_label) in zip(axes, panels):
    ax.axhline(reference, color=COLORS["muted"], lw=1.2, ls=":", zorder=1, label=reference_label)
    for name, (color, linestyle) in styles.items():
        table = np.array([summary[n][name][:, column] for n in SIZES])  # (sizes, repetitions)
        ax.fill_between(SIZES, table.min(axis=1), table.max(axis=1), color=color, alpha=0.18,
                        zorder=2)
        ax.plot(SIZES, table.mean(axis=1), color=color, ls=linestyle, lw=2.0, marker="o",
                ms=5, zorder=3, label=name)
    ax.set_xscale("log")
    ax.set_xlabel("samples per ensemble")
    ax.set_ylabel(ylabel)
    ax.set_ylim(-0.03, 1.03)
axes[1].legend(loc="center right", fontsize=8)
fig.suptitle("Samples needed to tell the two ensembles apart, by weighting", y=1.0)
fig.tight_layout()
plt.show()

**Read the figure.** The same numbers as curves, AUC on the left and detection rate on the right, against the number of samples on a logarithmic axis. The legend of the right panel serves both. Each shaded band runs from the lowest to the highest of the five repetitions. The grey curve, the equal-weight QMI, climbs slowly and has not reached the dotted $0.8$ line of the right panel at 20,000 samples. The blue curve, with the first harmonic at $0.5$, reaches it at 200 samples. The rose curve (weight $0.2$) and the dashed dark curve (weight $0$) lie on the violet curve of the second-harmonic PLV and partly hide it. Lowering the weight moves the quantum measure leftwards, towards the classical curve, and the curves stop there.

## 3. Why it works, and where it stops

On a longer and longer record a statistic settles to a fixed value: its **long-run** value. For these embeddings the long-run QMI depends on the data through two numbers only, the circular moments of the phase difference that `04/16` used: the first, $m_1 = \langle e^{i\Delta\theta}\rangle$, whose modulus is the PLV, and the second, $m_2 = \langle e^{2i\Delta\theta}\rangle$, whose modulus is the second-harmonic PLV. In our ensembles both are real and positive. `long_run_statistics` computes the long-run value from the moments themselves, with no sampling, so we can draw how each weighting responds to each moment.

Our comparison asks one thing of a measure: respond to $m_2$, which goes from $0$ to $0.3$. It asks it *not* to respond to $m_1$, which is $0.4$ in both ensembles and wanders around that value on any finite record. The next cell draws both responses.

In [ ]:
# long_run_statistics computes the measure from the moments themselves (the limit of an
# infinitely long record), so these curves carry no sampling noise. Moments are passed as
# (1, m1, m2), here real and positive.
WEIGHT_SETS = {
    "(1, 1, 1)": (1.0, 1.0, 1.0),
    "(1, 0.5, 2)": (1.0, 0.5, 2.0),
    "(1, 0.2, 2)": (1.0, 0.2, 2.0),
    "(1, 0, 2)": (1.0, 0.0, 2.0),
}


def long_run_qmi(m1, m2):
    """Long-run QMI (nats) of every weighting at first moment m1 and second moment m2."""
    stats = long_run_statistics(np.array([1.0, m1, m2], dtype=complex), weight_sets=WEIGHT_SETS)
    return np.array([stats[name] for name in WEIGHT_SETS])


m2_grid = np.linspace(0.0, 0.4, 41)
m1_grid = np.linspace(0.3, 0.5, 41)
along_m2 = np.array([long_run_qmi(0.4, m2) for m2 in m2_grid])   # (41, weightings)
along_m1 = np.array([long_run_qmi(m1, 0.15) for m1 in m1_grid])  # (41, weightings)

# Check against the ensembles of section 1: the long-run gap between m2 = 0 and m2 = 0.3.
print("long-run gap between m2 = 0.0 and m2 = 0.3 at m1 = 0.4, beside the gap of section 1")
for j, name in enumerate(WEIGHT_SETS):
    sampled = f"{gaps[j]:.5f}" if j < len(gaps) else "    -"
    print(f"  {name:<12} long run {along_m2[30, j] - along_m2[0, j]:.5f}   sampled {sampled}")

# The scale of the weighted measures near zero: double m2 and see by how much the long-run
# QMI, counted from its value at m2 = 0, is multiplied. A square law gives 4.
print("\nlong-run QMI counted from m2 = 0, at m1 = 0.4: effect of doubling m2")
print(f"{'weights':<14}{'m2 = 0.05':>11}{'m2 = 0.10':>11}{'m2 = 0.20':>11}{'ratios':>14}")
for j, name in enumerate(WEIGHT_SETS):
    rise = along_m2[[5, 10, 20], j] - along_m2[0, j]
    print(f"{name:<14}{rise[0]:>11.5f}{rise[1]:>11.5f}{rise[2]:>11.5f}"
          f"{rise[1] / rise[0]:>8.1f}{rise[2] / rise[1]:>6.1f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
curve_colors = [COLORS["muted"], COLORS["quantum"], COLORS["highlight"], COLORS["text"]]
curve_styles = ["-", "-", "-", "--"]
for j, name in enumerate(WEIGHT_SETS):
    axes[0].plot(m2_grid, along_m2[:, j] - along_m2[0, j], color=curve_colors[j],
                 ls=curve_styles[j], lw=2.0, label=name)
    axes[1].plot(m1_grid, along_m1[:, j] - along_m1[20, j], color=curve_colors[j],
                 ls=curve_styles[j], lw=2.0, label=name)
axes[0].axhline(0.0, color=COLORS["muted"], lw=1.0, zorder=1)
axes[1].axhline(0.0, color=COLORS["muted"], lw=1.0, zorder=1)
axes[0].set_xlabel("second moment $m_2$  (first moment fixed at 0.4)")
axes[0].set_ylabel("QMI minus its value at $m_2 = 0$  (nats)")
axes[0].set_title("What the comparison asks for", pad=10)
axes[1].set_xlabel("first moment $m_1$  (second moment fixed at 0.15)")
axes[1].set_ylabel("QMI minus its value at $m_1 = 0.4$  (nats)")
axes[1].set_title("What the comparison does not ask for", pad=10)
axes[0].legend(title="weights", loc="upper left", fontsize=9)
fig.suptitle("Long-run response of each weighting to the two moments", y=1.0)
fig.tight_layout()
plt.show()

**Read the output and the figure.** The first block checks the tool against section 1: the long-run gaps agree with the sampled ones within $0.00015$ nat for the weighted embeddings. The equal-weight one is off by $0.0005$ nat, and the next cell shows why that embedding is the one a finite record disturbs most. The second block checks the scale announced in section 2: for the weightings $(1, 0.2, 2)$ and $(1, 0, 2)$, doubling $m_2$ multiplies the rise of the measure by $4.0$, which is what a square law gives.

*Left panel.* As $m_2$ grows with $m_1$ fixed, the three weighted curves rise from the start, and the rose and dashed ones nearly coincide. The grey equal-weight curve first goes *down*, stays below its starting value until $m_2 \approx 0.17$, and only then rises. It is the curve that `04/16` drew in answer to the Challenge of `04/15`. A measure that first moves the wrong way spends part of the difference it is given on getting back to where it started.

*Right panel.* As $m_1$ moves with $m_2$ fixed at $0.15$, the equal-weight curve is steep and the blue one much flatter. The rose curve is almost flat, and nearly hidden under the dashed one, which is exactly flat. The next cell turns this panel into numbers.

In [ ]:
# Slopes: how many nats the measure moves per unit of each moment, around m1 = 0.4, m2 = 0.15,
# from a step of 0.01 on each side.
slope_m2 = (long_run_qmi(0.4, 0.16) - long_run_qmi(0.4, 0.14)) / 0.02
slope_m1 = (long_run_qmi(0.41, 0.15) - long_run_qmi(0.39, 0.15)) / 0.02
print("slope of the measure at m1 = 0.4, m2 = 0.15  (nats per unit of moment)")
print(f"{'weights':<14}{'along m1':>10}{'along m2':>10}{'ratio m1 / m2':>16}")
for j, name in enumerate(WEIGHT_SETS):
    print(f"{name:<14}{slope_m1[j]:>10.4f}{slope_m2[j]:>10.4f}{slope_m1[j] / slope_m2[j]:>16.2f}")

# By hand: the sample PLV of a finite record wanders around 0.4. Multiply its scatter by the
# slope along m1 to get the scatter this wandering alone puts into each measure. Then measure
# the scatter of each measure over the same records, low and high, and set the larger of the
# two beside the long-run gap.
N_CHECK = 2_000
records = [
    matched_plv_ensembles(N_CHECK, a1=0.4, a2_low=0.0, a2_high=0.3, seed=int(seed))
    for seed in seed_table[SIZES.index(N_CHECK), 0, 0]
]  # 100 (low, high) pairs of ensembles of 2,000 samples
plv_scatter = np.std([plv(*low) for low, _ in records], ddof=1)
print(f"\nscatter of the sample PLV over {len(records)} low ensembles of {N_CHECK:,} samples: "
      f"{plv_scatter:.4f}")
print(f"{'':<14}{'scatter from':>14}{'measured scatter':>20}{'long-run':>12}{'gap / larger':>14}")
print(f"{'weights':<14}{'the PLV':>14}{'low':>10}{'high':>10}{'gap':>12}{'scatter':>14}")
for j, (name, weights) in enumerate(WEIGHT_SETS.items()):
    from_plv = slope_m1[j] * plv_scatter
    low_scatter = np.std([weighted_qmi(*low, weights) for low, _ in records], ddof=1)
    high_scatter = np.std([weighted_qmi(*high, weights) for _, high in records], ddof=1)
    gap = along_m2[30, j] - along_m2[0, j]
    print(f"{name:<14}{from_plv:>14.5f}{low_scatter:>10.5f}{high_scatter:>10.5f}{gap:>12.5f}"
          f"{gap / max(low_scatter, high_scatter):>14.1f}")

**Read the output.** The **slope** of a measure along a moment is how many nats it moves when that moment moves by one unit; we read it from a small step on each side of the point $m_1 = 0.4$, $m_2 = 0.15$. The **scatter** of a quantity is its standard deviation over the hundred records.

The first table gives the slopes. The equal-weight measure moves by $0.37$ nat per unit of $m_1$, about twenty-three times its slope along $m_2$. With the weight at $0.2$ the slope along $m_1$ is $0.007$, a sixth of the slope along $m_2$. With the weight at $0$ it is zero.

The second table is a computation by hand, set beside a measurement. The sample PLV of a 2,000-sample record scatters by $0.012$ around $0.4$. Multiplied by the slope, a linear approximation taken at the midpoint between the two ensembles, that wandering alone puts $0.0046$ nat of scatter into the equal-weight QMI. The measured scatter is $0.0047$ nat for the low ensembles and $0.0056$ for the high ones: most of it comes from the matched moment, and it is as large as the whole gap we are after ($0.0049$).

For $(1, 0.2, 2)$ the first moment passes on $0.00009$ nat only, and the measured scatter is $0.00015$ nat for the low ensembles and $0.0015$ for the high ones. What remains is the scatter of the second moment itself, larger where that moment is not zero. No weighting removes it: it belongs to the moment we want to read. The gap is now about ten times the larger scatter, where it was a little under once. The equal-weight measure listens mostly to a moment that is the same in both ensembles, and the weights let you turn that moment down.

**Where it stops.** Follow the weight to zero. The long-run measure then depends on $m_2$ alone and rises with it: it is the second-harmonic PLV, read on another scale. A change of scale cannot rank two records differently, so it cannot raise the AUC.

On a finite record the match is close and not exact. The matrix also holds sample moments that vanish only in the long run: those of each channel on its own, and that of the sum of the two phases. With 50 samples they are not negligible, which is why the AUC of the $(1, 0, 2)$ row is $0.94$ where the PLV has $0.96$.

Turning the dial carries the quantum measure *to* the classical statistic. On these ensembles it has nowhere further to go.

## 4. Another lever, beside its classical counterpart

Weighting is not the only dial. Any other lever is scored the same way, with the protocol, beside the classical statistic built for the same job; the second exercise does it for more harmonics. Here is a lever of a different kind.

The amplitude embedding $\cos\alpha\,|0\rangle + \sin\alpha\,e^{i\theta}|1\rangle$, with $\alpha = \tfrac{\pi}{2}\,\mathrm{amp}$, takes a second input: `amp`, the amplitude of the signal, scaled to lie between 0 and 1. The weight of $|1\rangle$ then follows the amplitude sample by sample. The embedding can therefore respond to **phase–amplitude coupling**, in which the amplitude of an oscillation depends on a phase (Canolty & Knight 2010). PLV, which never sees an amplitude, cannot.

We build two kinds of dyad with the same phases, hence the same PLV, and the same amplitude values. In the **coupled** kind the amplitude follows the phase difference, $0.6 + 0.4\cos\Delta\theta$. In the **shuffled** kind the same amplitude record is shuffled in time, which breaks its tie to the phase and changes nothing else. Channel B gets its own amplitude record: the same values, shuffled independently.

The classical statistic built for this job is the **modulation index** of Tort et al. (2010): bin the phase, take the mean amplitude in each bin, and measure how far that profile is from flat with the Kullback–Leibler divergence of module 02. The index was defined for the phase of one oscillation and the amplitude of another; we adapt it here, with the phase difference of the dyad as the phase and channel A's amplitude as the amplitude. First, one large dyad of each kind.

In [ ]:
def draw_dyad(n, coupled, rng):
    """One dyad of n samples: phases with first moment 0.4, and one amplitude record per channel.

    Channel A's amplitude follows the phase difference. A coupled dyad uses it as it is; a
    control dyad shuffles it in time (same values, tie to the phase broken). Channel B gets an
    independent shuffle of the same values in both kinds, so the two amplitude records are unrelated.
    """
    theta_a = rng.uniform(0, 2 * np.pi, n)
    delta = sample_phase_difference(n, a1=0.4, a2=0.0, seed=int(rng.integers(2**31)))
    theta_b = np.mod(theta_a - delta, 2 * np.pi)
    amp_a = 0.6 + 0.4 * np.cos(theta_a - theta_b)  # amplitude follows the phase difference
    amp_b = rng.permutation(amp_a)                 # channel B: same values, unrelated order
    if not coupled:
        amp_a = rng.permutation(amp_a)             # same values, order unrelated to the phases
    return theta_a, theta_b, amp_a, amp_b


def amplitude_qmi(theta_a, theta_b, amp_a, amp_b):
    """QMI (nats) of the amplitude embedding cos(alpha)|0> + sin(alpha) e^{i theta}|1>."""
    rho = joint_density_from_states(
        amplitude_phase_state(theta_a, amp_a), amplitude_phase_state(theta_b, amp_b)
    )
    return coupling_qmi(rho, dims=(2, 2))


def tort_modulation_index(theta_a, theta_b, amp_a, amp_b, n_bins=18):
    """Modulation index of Tort et al. (2010), with the phase difference as the phase.

    Bins the phase, takes the mean amplitude of channel A per bin, normalises it to a
    distribution over bins, and returns its KL divergence from uniform divided by log(n_bins),
    in [0, 1]. Channel B's amplitude is not used.
    """
    edges = np.linspace(-np.pi, np.pi, n_bins + 1)
    wrapped = np.angle(np.exp(1j * (theta_a - theta_b)))  # wrap to [-pi, pi)
    idx = np.clip(np.digitize(wrapped, edges) - 1, 0, n_bins - 1)
    mean_amp = np.array([amp_a[idx == j].mean() if np.any(idx == j) else 0.0 for j in range(n_bins)])
    p = mean_amp / mean_amp.sum()
    return kl_divergence(p, np.full(n_bins, 1.0 / n_bins), base=np.e) / np.log(n_bins)


rng = np.random.default_rng(7)
shuffled, coupled = draw_dyad(N, False, rng), draw_dyad(N, True, rng)
print(f"{'':<26}{'shuffled':>13}{'coupled':>11}{'difference':>13}")
for name, statistic in (
    ("PLV", lambda ta, tb, amp_a, amp_b: plv(ta, tb)),
    ("Tort modulation index", tort_modulation_index),
    ("amplitude QMI (nats)", amplitude_qmi),
):
    a, b = statistic(*shuffled), statistic(*coupled)
    print(f"{name:<26}{a:>13.5f}{b:>11.5f}{b - a:>+13.5f}")

**Read the output.** The PLV is $0.40$ in both dyads, as built. The modulation index is zero to five decimals for the shuffled amplitude and $0.040$ for the coupled one. The amplitude QMI moves too, from $0.00340$ to $0.00004$ nat, and notice the direction: it goes *down* when the amplitude is tied to the phase. Each qubit's coherence carries $\sin\alpha\cos\alpha = \tfrac12\sin(\pi\,\mathrm{amp})$, zero at $\mathrm{amp} = 1$, and in the coupled dyad A's amplitude nears 1 where the phase difference gathers, which hides the phase coupling. The embedding decides in which direction the measure responds, so its value is not a "more coupling, more information" scale, and the detection band of the protocol has two sides: a difference of either sign counts. The protocol now tells at which record length each statistic separates them.

In [ ]:
AMPLITUDE_STATISTICS = {
    "PLV": lambda ta, tb, amp_a, amp_b: plv(ta, tb),
    "Tort modulation index": tort_modulation_index,
    "amplitude QMI": amplitude_qmi,
}
AMPLITUDE_SIZES = [200, 2_000, 20_000]

amplitude_summary = {}
for k, n in enumerate(AMPLITUDE_SIZES):
    per_repetition = {name: [] for name in AMPLITUDE_STATISTICS}
    for s in range(N_SEEDS):
        rng = np.random.default_rng([170, k, s])
        values = {name: np.empty((4, N_RECORDS)) for name in AMPLITUDE_STATISTICS}
        for i in range(N_RECORDS):
            # shuffled, coupled, and two more shuffled dyads for the null differences
            dyads = [draw_dyad(n, flag, rng) for flag in (False, True, False, False)]
            for name, statistic in AMPLITUDE_STATISTICS.items():
                values[name][:, i] = [statistic(*dyad) for dyad in dyads]
        for name, v in values.items():
            per_repetition[name].append((
                auc(v[0], v[1]),
                detection_rate(v[1] - v[0], v[3] - v[2], false_alarm_rate=FALSE_ALARM_RATE),
            ))
    amplitude_summary[n] = {name: np.array(v) for name, v in per_repetition.items()}
print_tables(amplitude_summary, AMPLITUDE_SIZES, "phase-amplitude coupling")

**Read the output.** The PLV stays at chance at every length: an AUC near $0.5$ and a detection rate between $0.03$ and $0.08$, around the false-alarm rate of $0.05$ we allowed.

The modulation index ranks the dyads almost perfectly from 200 samples on (AUC $0.99$), detects $0.79$ of them at 200 samples, and all from 2,000 on.

The amplitude QMI detects $0.32$ at 200 samples and all from 2,000 on. Its AUC is $0.04$, then $0.00$: a separation read in the other direction, coupled dyads scoring *below* shuffled ones. At 200 samples its detection rate ranges from $0.11$ to $0.72$ across repetitions while every AUC stays between $0.02$ and $0.06$: the band, estimated from a hundred null differences, moves, and the ranking does not depend on it.

The quantum measure reaches phase–amplitude coupling, behind the modulation index at 200 samples and level with it from 2,000 on, so on this grid it needs more samples than the statistic built for the job, and at most ten times more.

## 5. What the dial buys

The weights divided the number of samples the quantum measure needs by at least two hundred, and brought it level with the second-harmonic PLV, which it resembles more and more as the weight goes to zero. The amplitude embedding reached a coupling PLV cannot see, behind the modulation index on short records. Each lever changed how many samples the quantum measure needs. None made it read something the matching classical statistic misses, or rank the records better than that statistic does.

That is the conclusion of `04/16`, seen from the side of design. The embedding decides which classical statistic your measure resembles, so let that statistic guide the design: find the moment that carries your difference, and give it the weight. What the framework adds is the common construction: one density matrix and one coupling measure reach phase synchrony, a higher phase moment and phase–amplitude coupling by a change of embedding, where the classical toolbox has a separate statistic for each.

**Limits of what was measured here.** One pair of ensembles for the weights and the harmonics (first moment $0.4$, second moment $0$ against $0.3$) and one pair for the amplitude, in which only channel A's amplitude follows the phase. Samples drawn independently, where real signals are correlated in time; the bench of `04/16` covers that case for the weights. Four weightings, not an optimisation, picked knowing which moment differs and scored on the same ensembles; on your data, fix them from a hypothesis stated in advance, or on held-out records. The $0.8$ line and the two-or-three-bands rule are conventions. Notebook `04/18` asks whether *any* embedding of classical data could cross the boundary met here.

## Your turn

**Warm-up.** Add the weighting $(1, 0.1, 2)$ to `STATISTICS` and to `WEIGHT_SETS`. Before you run sections 2 and 3 again, predict between which two rows its detection rate will fall at 50 and 100 samples, and whether its slope along $m_1$ will be above or below $0.0074$. (It falls between the rows of the weights $0.2$ and $0$, which are already level, with a slope of about $0.002$: the factor $w_1^2$ of section 1 at work.)

**Go further.** Another lever: more harmonics. A *qudit* is a system with $d$ levels (the qutrit is $d = 3$), and `weighted_qmi(ta, tb, None, (1, 2, 3))` is the QMI of the equal-weight embedding with four levels. Our two ensembles have no third or fourth moment, so the added levels carry no new difference between them. Predict whether they help anyway, then add the embeddings with harmonics $(1, 2, 3)$ and $(1, 2, 3, 4)$ to `STATISTICS` and read the 2,000-sample column. (They do help: the detection rate goes from $0.15$ to about $0.31$ and $0.47$, and the AUC from $0.74$ to about $0.89$ and $0.94$, with the second-harmonic PLV at $1.00$ on both. With more levels, more pairs of levels sit two steps apart, and each carries the second moment: your numbers are consistent with that reading and do not isolate it as the cause.)

**Challenge.** You want to detect phase–amplitude coupling *and* a second-moment phase coupling at once. Sketch an embedding that carries both (hint: combine the amplitude weight of `amplitude_phase_state` with the harmonic stack of `weighted_multifreq_state`). Your sketch is complete when it names, for *each* coupling, a pair of ensembles that differ in that coupling alone, the classical statistic you would set beside the embedding, and the result on the two criteria of `04/15` that would let you write "level with the classical statistic".

## What you built

You turned the embedding into an instrument you can tune, and you measured every turn of the dial. That is careful work, and it is yours.

- You turned the first harmonic down with `weighted_multifreq_state`, and saw the gap between the ensembles of `04/15` grow from $0.0044$ to $0.0145$ nat.
- You scored each weighting with the protocol of `04/15` and found the number of samples needed **divided by at least two hundred**, down to the level of the second-harmonic PLV.
- You explained it from the long-run response and checked it on measured scatter: the equal-weight measure listens mostly to the matched moment, and at weight zero the long-run measure is the classical statistic on another scale.
- You learned to read a near-tie: the AUC ignores the scale of a statistic, the detection rate does not.
- You scored the amplitude embedding beside the modulation index: it reaches phase–amplitude coupling, behind the index on short records.

You can now tune an embedding on purpose, and say exactly what the tuning bought.

## What's next

Neither lever of this notebook took the quantum measure past the matching classical statistic. Is that boundary a habit of the embeddings we happened to try, or a fact about all of them? In `18_what_no_classical_data_embedding_can_buy` we put the sharpest candidate to the test, an embedding that *entangles* the two systems, and find out.

## References

- K. V. Mardia & P. E. Jupp, *Directional Statistics* (Wiley, 2000). DOI:10.1002/9780470316979 — circular moments; the weighting tunes which moment the coupling measure emphasises.
- J.-P. Lachaux, E. Rodriguez, J. Martinerie & F. J. Varela, "Measuring phase synchrony in brain signals", *Human Brain Mapping* **8**, 194–208 (1999). DOI:10.1002/(SICI)1097-0193(1999)8:4<194::AID-HBM4>3.0.CO;2-C — the phase-locking value, the first-moment statistic whose channel we turn down.
- M. M. Wilde, *Quantum Information Theory*, 2nd ed. (Cambridge University Press, 2017). DOI:10.1017/9781316809976 — quantum mutual information, the coupling measure read off $\rho_{AB}$.
- J. A. Hanley & B. J. McNeil, "The meaning and use of the area under a receiver operating characteristic (ROC) curve", *Radiology* **143**, 29 (1982). DOI:10.1148/radiology.143.1.7063747 — the area under the curve, read as a probability of correct ranking.
- A. B. L. Tort, R. Komorowski, H. Eichenbaum & N. Kopell, "Measuring phase-amplitude coupling between neuronal oscillations of different frequencies", *Journal of Neurophysiology* **104**, 1195–1210 (2010). DOI:10.1152/jn.00106.2010 — the modulation index, the classical statistic set beside the amplitude embedding.
- R. T. Canolty & R. T. Knight, "The functional role of cross-frequency coupling", *Trends in Cognitive Sciences* **14**, 506–515 (2010). DOI:10.1016/j.tics.2010.09.001 — review of phase–amplitude coupling as a studied neural phenomenon.

**Previous:** `notebooks/04_QuantumOptimalTransport/16_capstone_synthesis.ipynb` · **Next:** `notebooks/04_QuantumOptimalTransport/18_what_no_classical_data_embedding_can_buy.ipynb`